# Automação Web e Busca de Informações com Python

#### Desafio: 

Trabalhamos em uma importadora e o preço dos nossos produtos é vinculado a cotação de:
- Dólar
- Euro
- Ouro

Precisamos pegar na internet, de forma automática, a cotação desses 3 itens e saber quanto devemos cobrar pelos nossos produtos, considerando uma margem de contribuição que temos na nossa base de dados.

Base de Dados: https://drive.google.com/drive/folders/1KmAdo593nD8J9QBaZxPOG1yxHZua4Rtv?usp=sharing

Para isso, vamos criar uma automação web:

- Usaremos o selenium
- Importante: baixar o webdriver

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# Passo 1: Pegar a cotação do dólar
# abrir o navegador
navegador = webdriver.Chrome()
# entrar no google
navegador.get("https://www.google.com.br/")
# pesquisar no google por "cotação do dólar"
navegador.find_element(By.XPATH, '//*[@id="ti6dpd"]').send_keys("cotação do dólar")
navegador.find_element(By.XPATH, '//*[@id="ti6dpd"]').send_keys(Keys.ENTER)
input("⚠️ Resolva o CAPTCHA no navegador, volte aqui no terminal e pressione ENTER para continuar...")
# pegar a cotação que está no google
cotacao_dolar = navegador.find_element(By.XPATH, '//*[@id="knowledge-currency__updatable-data-column"]/div[1]/div[2]/span[1]').get_attribute("data-value")
print(f"Cotação do dólar: {cotacao_dolar}")

# Passo 2: Pegar a cotação do euro
navegador.get("https://www.google.com.br/")
# pesquisar no google por "cotação do euro"
navegador.find_element(By.XPATH, '//*[@id="ti6dpd"]').send_keys("cotação do euro")
navegador.find_element(By.XPATH, '//*[@id="ti6dpd"]').send_keys(Keys.ENTER)
# pegar a cotação que está no google
cotacao_euro = navegador.find_element(By.XPATH, '//*[@id="knowledge-currency__updatable-data-column"]/div[1]/div[2]/span[1]').get_attribute("data-value")
print(f"Cotação do euro: {cotacao_euro}")

# Passo 3: Pegar a cotação do ouro
navegador.get("https://www.melhorcambio.com/ouro-hoje")
cotacao_ouro = navegador.find_element(By.XPATH, '//*[@id="comercial"]').get_attribute("value")
cotacao_ouro = cotacao_ouro.replace(",", ".")
print(f"Cotação do ouro: {cotacao_ouro}")

Cotação do dólar: 5.0449
Cotação do euro: 5.627464
Cotação do ouro: 656.03


### Agora vamos atualiza a nossa base de preços com as novas cotações

- Importando a base de dados

In [2]:
import pandas as pd

# Passo 4: Atualizar a base de preços (atualizando o preço de compra e o de venda)
tabela = pd.read_excel("Produtos.xlsx")
display(tabela)

,Produtos,Preço Original,Moeda,Cotação,Preço de Compra,Margem,Preço de Venda
0,Câmera Canon,999.99,Dólar,5,4999.95,1.40,6999.930
1,Carro Renault,4500.00,Euro,6,27000.00,2.00,54000.000
2,Notebook Dell,899.99,Dólar,5,4499.95,1.70,7649.915
3,IPhone,799.00,Dólar,5,3995.00,1.70,6791.500
4,Carro Fiat,3000.00,Euro,6,18000.00,1.90,34200.000
5,Celular Xiaomi,480.48,Dólar,5,2402.40,2.00,4804.800
6,Joia 20g,20.00,Ouro,350,7000.00,1.15,8050.000


- Atualizando os preços e o cálculo do Preço Final

In [3]:
# atualizar a coluna de cotação
# editar a coluna cotação onde a coluna moeda = Dólar
# Alterar tipo da coluna para float
tabela["Cotação"] = tabela["Cotação"].astype(float)
tabela.loc[tabela["Moeda"] == "Dólar", "Cotação"] = float(cotacao_dolar)
# editar a coluna cotação onde a coluna moeda = Euro
tabela.loc[tabela["Moeda"] == "Euro", "Cotação"] = float(cotacao_euro)
# editar a coluna cotação onde a coluna moeda = Ouro
tabela.loc[tabela["Moeda"] == "Ouro", "Cotação"] = float(cotacao_ouro)
#tabela["Cotação"] = tabela["Cotação"].map("{:.2f}".format)

# atualizar a coluna de preço de compra = preço original * cotação
tabela["Preço de Compra"] = tabela["Preço Original"] * tabela["Cotação"]
#tabela["Preço de Compra"] = tabela["Preço de Compra"].map("{:.2f}".format)

# atualizar a coluna de preço de venda = preço de compra * margem
tabela["Preço de Venda"] = tabela["Preço de Compra"] * tabela["Margem"]
#tabela["Preço de Venda"] = tabela["Preço de Venda"].map("{:.2f}".format)

In [4]:
display(tabela)

,Produtos,Preço Original,Moeda,Cotação,Preço de Compra,Margem,Preço de Venda
0,Câmera Canon,999.99,Dólar,5.044900,5044.849551,1.40,7062.789371
1,Carro Renault,4500.00,Euro,5.627464,25323.588000,2.00,50647.176000
2,Notebook Dell,899.99,Dólar,5.044900,4540.359551,1.70,7718.611237
3,IPhone,799.00,Dólar,5.044900,4030.875100,1.70,6852.487670
4,Carro Fiat,3000.00,Euro,5.627464,16882.392000,1.90,32076.544800
5,Celular Xiaomi,480.48,Dólar,5.044900,2423.973552,2.00,4847.947104
6,Joia 20g,20.00,Ouro,656.030000,13120.600000,1.15,15088.690000


### Agora vamos exportar a nova base de preços atualizada

In [ ]:
# Passo 5: Exportar a base de preços atualizado
tabela.to_excel("Produtos_atualizado.xlsx", index=False)
navegador.quit()